## Generating  Customers Data 


In [27]:
import pandas as pd
import random
from faker import Faker

fake = Faker("en_NG")
num_records = 1000

data = []

# Valid Nigerian prefixes
valid_prefixes = ["070", "080", "081", "090", "091", "071"]

def generate_valid_phone():
    prefix = random.choice(valid_prefixes)
    remaining_digits = "".join(str(random.randint(0, 9)) for _ in range(8))
    return prefix + remaining_digits  # total = 11 digits

for _ in range(num_records):
    record = {
        "name": fake.name() if random.random() > 0.1 else None,  # 10% missing

        # ✅ Phone numbers: valid or missing only
        "phone_number": (
            generate_valid_phone() if random.random() > 0.1 else None
        ),

        # IDs: mostly valid, some invalid
        "id_number": (
            str(fake.random_number(digits=11, fix_len=True))
            if random.random() > 0.2
            else str(fake.random_number(digits=random.choice([8, 9, 12, 13])))
        ),

        "address": fake.address() if random.random() > 0.1 else None,

        "reg_date": (
            fake.date_between(start_date="-2y", end_date="today")
            if random.random() > 0.1 else None
        )
    }

    data.append(record)

df_raw = pd.DataFrame(data)

# Save raw data
df_raw.to_csv("sim_registration_raw.csv", index=False)

print("Raw fake data created successfully")
print(df_raw.head(10))


Raw fake data created successfully
                 name phone_number    id_number  \
0                None  08043217414  80462493492   
1       Sarah Ekwueme  09157145941  45894144782   
2                None  07191348731  61499973212   
3                None         None  89892013535   
4  Cornelius Obasanjo  08103832158    706276592   
5     Charity Ibrahim  08060937156  65175471759   
6         Mary Okafor  07044735468  91594987722   
7      Philip Adeyemi  09185341629  64751744371   
8                None         None  98687490344   
9       Daniel Okafor  09048793192  85317019711   

                                             address    reg_date  
0   5875 Joshua Spurs Apt. 288\nOshoditown, LA 35540  2025-11-09  
1                                               None  2024-08-21  
2  49282 Samuel Isle Suite 078\nNorth Cornelius, ...  2025-12-17  
3      13583 Akinwale Valleys\nGabrielberg, SD 24854  2025-07-11  
4                                               None  2025-10-06  
5

In [28]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          895 non-null    object
 1   phone_number  910 non-null    object
 2   id_number     1000 non-null   object
 3   address       923 non-null    object
 4   reg_date      901 non-null    object
dtypes: object(5)
memory usage: 39.2+ KB


## ETL Pipeline

In [ ]:
import pandas as pd
import sqlite3
import re

# -------------------------
# 1. EXTRACTION
# -------------------------
print("\n1️⃣ EXTRACTING DATA...")

csv_path = "sim_registration_raw.csv"

df = pd.read_csv(
    csv_path,
    dtype={"phone_number": str, "id_number": str},
    na_values=["", "NaN", "NA", "None", "nan"]
)


print(f"Rows extracted: {len(df):,}")
print(df.head())

# -------------------------
# 2. TRANSFORMATION
# -------------------------
print("\n2️⃣ TRANSFORMING DATA...")

# -------- NAME CLEANING --------
df["name"] = (
    df["name"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# -------- PHONE NUMBER CLEANING --------
def clean_phone(x):
    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove trailing .0 if it exists
    if x.endswith(".0"):
        x = x[:-2]

    digits = re.sub(r"\D", "", x)

    # Convert Nigerian country code to local
    if digits.startswith("234") and len(digits) > 3:
        digits = "0" + digits[3:]

    if len(digits) == 11 and digits.startswith("0"):
        return digits

    return None

df["phone_number"] = df["phone_number"].apply(clean_phone)
df = df.dropna(subset=["phone_number"])

# -------- ID NUMBER CLEANING --------
import re
import pandas as pd
import numpy as np

def clean_id(x):
    if pd.isna(x):
        return None

    digits = re.sub(r"\D", "", str(x))

    # Keep only valid 11-digit IDs
    if len(digits) == 11:
        return digits

    return None

# Apply cleaning
df["id_number"] = df["id_number"].apply(clean_id)

# Drop unknown / invalid IDs
df = df.dropna(subset=["id_number"])


# -------- ADDRESS CLEANING --------
df["address"] = (
    df["address"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# -------- REGISTRATION DATE --------
df["reg_date"] = pd.to_datetime(
    df["reg_date"],
    errors="coerce"
)
# fill empty dates with placeholder (1900-01-01)
df["reg_date"] = df["reg_date"].fillna(pd.Timestamp("1900-01-01"))


print("Transformation completed")

# -------------------------
# 3. LOAD (CSV)
# -------------------------
print("\n3️⃣ LOADING CLEAN DATA TO CSV...")

clean_csv = "sim_registration_cleaned.csv"
df.to_csv(clean_csv, index=False)

print(f"Clean CSV saved: {clean_csv}")
print(f"Final rows: {len(df):,}")

# -------------------------
# 4. LOAD (DATABASE)
# -------------------------
print("\n4️⃣ LOADING DATA TO DATABASE...")

conn = sqlite3.connect("sim_registration.db")

df.to_sql(
    "sim_registrations",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Data successfully loaded into SQLite database")



1️⃣ EXTRACTING DATA...
Rows extracted: 1,000
              name phone_number    id_number  \
0     Sarah Abiola  08003804687  48528995637   
1    Andrew Oshodi  07115068455  63117217589   
2   Agnes Ogunleye  09184097450     91139355   
3        Peace Eze          NaN  57995870179   
4  Charity Balogun  08017307063  98076699451   

                                             address    reg_date  
0       4995 Chukwu Hill\nNorth Angelastad, HI 31151  2025-11-14  
1        08649 David Motorway\nOkaforville, VT 88164  2025-02-20  
2  8515 Peter Ways Apt. 738\nSouth Maryfort, TX 1...  2024-02-08  
3     7071 Hope Ways Apt. 391\nPort Andrew, MD 22040  2024-09-22  
4    95678 Obi Fields\nEast Nathanielville, NC 41318  2025-04-19  

2️⃣ TRANSFORMING DATA...
Transformation completed

3️⃣ LOADING CLEAN DATA TO CSV...
Clean CSV saved: sim_registration_cleaned.csv
Final rows: 707

4️⃣ LOADING DATA TO DATABASE...
Data successfully loaded into SQLite database


In [23]:
df = pd.read_csv("sim_registration_cleaned.csv")
df

,name,phone_number,id_number,address,reg_date
0,Sarah Abiola,8003804687,48528995637,"4995 Chukwu Hill\nNorth Angelastad, HI 31151",2025-11-14
1,Andrew Oshodi,7115068455,63117217589,"08649 David Motorway\nOkaforville, VT 88164",2025-02-20
2,Charity Balogun,8017307063,98076699451,"95678 Obi Fields\nEast Nathanielville, NC 41318",2025-04-19
3,Cornelius Olawale,8010440331,48227740487,Unknown,1900-01-01
4,Juliet Chukwu,9057421361,13940979072,"866 Chukwu Isle Suite 253\nPeaceport, VI 53297",2025-06-30
...,...,...,...,...,...
702,Emmanuel Obi,9011355629,94958867277,"9974 Adetokunbo Knolls\nNnamanichester, AZ 75730",2024-02-29
703,Unknown,8169690626,71396430580,USCGC Abiola\nFPO AE 98295,2025-11-14
704,Juliet Akinwale,7106717025,98573128468,"00006 Emmanuel Dam Suite 700\nObiville, CO 56127",1900-01-01
705,Peace Okonkwo,8028588296,86306727833,"72652 Hope Forks\nGabrielberg, MA 46087",2025-11-15


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          707 non-null    object
 1   phone_number  707 non-null    int64 
 2   id_number     707 non-null    int64 
 3   address       707 non-null    object
 4   reg_date      707 non-null    object
dtypes: int64(2), object(3)
memory usage: 27.7+ KB
